# 03 — Deployment Smoke Test Script

Companion notebook to chapters 02 and 04. A **smoke test** is the pipeline's last line of defense: a fast, shallow check that hits the newly deployed service's `/health` endpoint (with retries and backoff, since a service can take a few seconds to finish starting after a slot swap or container restart) before the pipeline declares the deployment successful. This is the same step referenced as `python scripts/smoke_test.py --url ...` in chapter 02's worked pipeline YAML.

This notebook runs **fully offline** — `unittest.mock` stands in for real HTTP calls, so no live server or network access is required to exercise the retry/backoff logic and see both the pass and fail code paths.

In [1]:
import time
from unittest.mock import MagicMock


class HttpResponse:
    """Minimal stand-in for an HTTP response object (e.g. requests.Response)."""

    def __init__(self, status_code, body=""):
        self.status_code = status_code
        self.body = body

    def __repr__(self):
        return f"HttpResponse(status_code={self.status_code!r})"

## The smoke test function

`smoke_test` takes an injectable `http_get` callable rather than hardcoding a real HTTP client — this is what makes it trivially mockable for this notebook, and is also just good design for a script meant to be unit-tested in the pipeline repo itself. It retries with **exponential backoff** (`backoff_seconds * 2 ** attempt`), which spreads retries out over time instead of hammering a service that's still warming up.

In [2]:
def smoke_test(url, http_get, max_retries=3, backoff_seconds=0.05, timeout=5):
    """Post-deploy smoke test: GET `url`, retrying with exponential backoff.

    Parameters
    ----------
    url : str
        The health-check (or other shallow) endpoint to verify, e.g.
        ``https://document-uploader-dev.azurewebsites.net/health``.
    http_get : callable(url, timeout) -> HttpResponse
        Injected so tests (and this notebook) can substitute a mock instead of a
        real network call. In a real pipeline script this would default to a
        ``requests.get``-based implementation.
    max_retries : int
        Number of attempts before giving up and reporting failure.
    backoff_seconds : float
        Base delay between retries; doubles after every failed attempt.

    Returns
    -------
    (passed: bool, message: str)
    """
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            response = http_get(url, timeout)
        except Exception as exc:
            last_error = f"attempt {attempt}: connection error: {exc}"
        else:
            if response.status_code == 200:
                return True, f"PASS: {url} returned 200 on attempt {attempt}"
            last_error = f"attempt {attempt}: got status {response.status_code}"
        if attempt < max_retries:
            time.sleep(backoff_seconds * (2 ** (attempt - 1)))
    return False, (
        f"FAIL: {url} did not return 200 after {max_retries} attempts. "
        f"Last: {last_error}"
    )

## Scenario 1 — healthy deploy, passes immediately

The mocked `http_get` returns `200` on the very first call, the way a normal, successful deployment should behave.

In [3]:
mock_get_ok = MagicMock(return_value=HttpResponse(200, "ok"))

passed, message = smoke_test("https://document-uploader-dev.azurewebsites.net/health", mock_get_ok)
print(passed, message)
print("HTTP calls made:", mock_get_ok.call_count)
assert passed is True
assert mock_get_ok.call_count == 1

True PASS: https://document-uploader-dev.azurewebsites.net/health returned 200 on attempt 1
HTTP calls made: 1


## Scenario 2 — slow cold start, passes after retries

The first call raises a connection error (the app is still starting), the second returns `503 Service Unavailable`, and the third succeeds — a realistic pattern right after a slot swap or container restart. This is exactly why the smoke-test step needs retries with backoff rather than a single one-shot check: a single failed request right after deploy doesn't necessarily mean the deployment is bad, it might just mean the service hasn't finished warming up yet.

In [4]:
mock_get_flaky = MagicMock(
    side_effect=[
        ConnectionError("cold start"),
        HttpResponse(503),
        HttpResponse(200, "ok"),
    ]
)

passed, message = smoke_test(
    "https://document-uploader-dev.azurewebsites.net/health",
    mock_get_flaky,
    max_retries=3,
)
print(passed, message)
print("HTTP calls made:", mock_get_flaky.call_count)
assert passed is True
assert mock_get_flaky.call_count == 3

True PASS: https://document-uploader-dev.azurewebsites.net/health returned 200 on attempt 3
HTTP calls made: 3


## Scenario 3 — genuinely broken deploy, fails after exhausting retries

The mocked service returns `500 Internal Server Error` on every attempt. After `max_retries` attempts, `smoke_test` gives up and reports failure — this is the signal a pipeline should treat as **"trigger rollback"** (chapter 04: swap the deployment slot back, or redeploy the previous known-good artifact tag) rather than declaring the release successful.

In [5]:
mock_get_down = MagicMock(return_value=HttpResponse(500))

passed, message = smoke_test(
    "https://document-uploader-dev.azurewebsites.net/health",
    mock_get_down,
    max_retries=3,
)
print(passed, message)
print("HTTP calls made:", mock_get_down.call_count)
assert passed is False
assert mock_get_down.call_count == 3

False FAIL: https://document-uploader-dev.azurewebsites.net/health did not return 200 after 3 attempts. Last: attempt 3: got status 500
HTTP calls made: 3


## Wiring this into a real pipeline

In an actual pipeline script (`scripts/smoke_test.py`, referenced from chapter 02's YAML), `http_get` would default to a small wrapper around `requests.get` (or `urllib.request` to avoid an extra dependency), the script would read the target URL from a CLI argument or pipeline variable, and it would `sys.exit(1)` on failure so the pipeline step itself reports as failed — which is what actually blocks promotion to the next environment or triggers the rollback path.

```python
# scripts/smoke_test.py (sketch -- not part of this notebook's execution)
import argparse, sys, urllib.request

def real_http_get(url, timeout):
    with urllib.request.urlopen(url, timeout=timeout) as resp:
        return HttpResponse(resp.status)

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--url', required=True)
    args = parser.parse_args()
    passed, message = smoke_test(args.url + '/health', real_http_get)
    print(message)
    sys.exit(0 if passed else 1)
```